In [1]:
import os
import json
import time
import requests
import pandas as pd
from tqdm import tqdm
from readability import Document
from bs4 import BeautifulSoup


In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}


In [ ]:
data_path = r"E:/GDELP/task_3/mideast&asia.csv"
df = pd.read_csv(data_path)

print(len(df))
print(df.columns)


10923
Index(['url', 'url_mobile', 'title', 'seen_date', 'image_url', 'domain',
       'language', 'source_country', 'culture_category'],
      dtype='object')


In [4]:
def extract_article_text(url, timeout=10):
    try:
        resp = requests.get(url, headers=headers, timeout=timeout)
        if resp.status_code != 200:
            return None, f"HTTP {resp.status_code}"

        html = resp.text
        doc = Document(html)
        clean_html = doc.summary(html_partial=True)

        soup = BeautifulSoup(clean_html, "lxml")
        paragraphs = [p.get_text(strip=True) for p in soup.find_all("p")]
        text = "\n".join(paragraphs)

        if len(text) < 200:  # 太短的正文基本是失败
            return None, "Text too short"

        return text, None

    except Exception as e:
        return None, str(e)


In [5]:
output_dir = r"E:/GDELP/task_3/articles_texts"
log_dir = r"E:/GDELP/task_3/logs"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

start_id = 25774
failed = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    url = row["url"]
    # # 第一次添加文章
    # article_id = f"{idx:06d}"

    # 后续添加文章
    article_id = f"{start_id + idx:06d}"

    save_path = os.path.join(output_dir, f"{article_id}.json")

    # 已存在就跳过（支持断点续跑）
    if os.path.exists(save_path):
        continue

    text, error = extract_article_text(url)

    if text is None:
        failed.append((article_id, url, error))
        continue

    article_data = {
        "article_id": article_id,
        "url": url,
        "text": text
    }

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(article_data, f, ensure_ascii=False, indent=2)

    time.sleep(0.5)  # 防止被封
failed_path = os.path.join(log_dir, "failed_urls.txt")

with open(failed_path, "w", encoding="utf-8") as f:
    for item in failed:
        f.write("\t".join(item) + "\n")

print(f"失败数量: {len(failed)}")







100%|██████████| 10923/10923 [14:25:23<00:00,  4.75s/it]      

失败数量: 3997


### 根据原csv的id顺序分配text_path，因此csv一定要是原csv，不能是筛选过图片后的csv

In [10]:
data_path = r"mideast&asia.csv.backup"
df = pd.read_csv(data_path)


# 指定项目根目录（包含 articles_texts 的父目录）
base_dir = r"E:/GDELP/task_3"
articles_texts_dir = os.path.join(base_dir, "articles_texts")
start_id = 25774

# 确保目录存在（可选，用于调试）
assert os.path.isdir(articles_texts_dir), f"articles_texts 目录不存在: {articles_texts_dir}"

# 假设 df 是你已有的 DataFrame（包含原始列）
df["article_id"] = [f"{start_id + i:06d}" for i in range(len(df))]

# 使用全路径生成 text_path
df["text_path"] = df["article_id"].apply(
    lambda x: os.path.join(articles_texts_dir, f"{x}.json")
)

# 过滤：只保留 text_path 对应文件实际存在的行
df = df[df["text_path"].apply(lambda path: os.path.exists(path))].reset_index(drop=True)

# 保存 CSV 到指定位置
output_csv_path = os.path.join(base_dir, "mideast&asia_with_textpath.csv")
df.to_csv(output_csv_path, index=False)

print(f"已保存 {len(df)} 条有效记录到 {output_csv_path}")



已保存 6926 条有效记录到 E:/GDELP/task_3\mideast&asia_with_textpath.csv


In [13]:
import pandas as pd

# 配置
csv_path = "mideast&asia_with_textpath.csv"         # ← 改成你的 CSV 文件路径
output_path = "mideast&asia_news_data.csv"   # 输出文件名
prefix = "E:/GDELP/task_3\\"         # 要去掉的前缀

# 读取
df = pd.read_csv(csv_path)

# 直接字符串替换（只替换开头匹配的部分）
df["image_path"] = df["image_path"].str.replace(prefix, "", n=1)
df["text_path"] = df["text_path"].str.replace(prefix, "", n=1)

# 保存
df.to_csv(output_path, index=False)
print(f"✅ 已保存到 {output_path}")

✅ 已保存到 mideast&asia_news_data.csv


In [14]:
# 将mideast&asia_news_data.csv合并进ur_conflict_news_data.csv
df_news = pd.read_csv(r"mideast&asia_news_data.csv")
df = pd.read_csv(r"ur_conflict_news_data.csv")
df = pd.concat([df, df_news], ignore_index=True)
df.to_csv(r"war_news_data.csv", index=False)


In [12]:
import pandas as pd

# 替换为你的 CSV 文件路径
input_file = "ur_conflict_news_data.csv"
output_file = "ur_conflict_news_data.csv"

# 读取数据
df = pd.read_csv(input_file)

# 精确替换（区分大小写）
df["sourcecountry"] = df["sourcecountry"].str.replace("United States Of America", "United States")

# 保存结果
df.to_csv(output_file, index=False)
print(f"✅ 已替换并保存到 {output_file}")

KeyError: 'sourcecountry'

In [15]:
# 3. 统计 sourcecountry 分布
df = pd.read_csv("war_news_data.csv")

print("=== source_country 分布 ===")
country_counts = df['source_country'].value_counts()
print(country_counts)
print(f"\n总共有 {df['source_country'].nunique()} 个不同的国家")

# 5. （可选）单独保存国家分布
country_counts.to_csv(r"E:/GDELP/task_3/sourcecountry_distribution.csv", header=["count"])

=== source_country 分布 ===
source_country
United States    5884
Russia           3506
Poland           3475
Germany          1964
India            1250
                 ... 
Nepal               1
Cuba                1
Ivory Coast         1
Netherlands         1
Oman                1
Name: count, Length: 87, dtype: int64

总共有 87 个不同的国家


In [ ]:
import pandas as pd
import shutil
from pathlib import Path

# 配置
csv_file = "ur_conflict_news_data.csv"
img_col = "image_path"
dst_dir = Path("ur_conflict_imgs_total_filtered")
dst_dir.mkdir(exist_ok=True)  # 自动创建文件夹

# 读取 CSV
df = pd.read_csv(csv_file)

# 遍历 image_path，复制存在的图片
copied = 0
for img_path_str in df[img_col].dropna():  # 跳过 NaN
    src = Path(img_path_str.strip())
    if src.is_file():
        dst = dst_dir / src.name
        shutil.copy2(src, dst)  # copy2 保留元数据
        copied += 1

print(f"✅ 共复制 {copied} 张图片到 {dst_dir}")

✅ 共复制 17058 张图片到 ur_conflict_imgs_total_filtered


In [2]:
import pandas as pd
import os

# ========== 配置参数 ==========
input_file = "sample.csv"   # 👈 替换为你的输入 CSV 文件路径
output_dir = "split_csv_output_sample"      # 👈 输出目录（可自定义）
rows_per_file = 20                 # 每个文件的行数

# ========== 创建输出目录 ==========
os.makedirs(output_dir, exist_ok=True)

# ========== 读取 CSV 文件 ==========
print(f"正在读取文件: {input_file}")
df = pd.read_csv(input_file)

total_rows = len(df)
print(f"总行数: {total_rows}")

# ========== 分割并保存 ==========
num_files = (total_rows // rows_per_file) + (1 if total_rows % rows_per_file != 0 else 0)

for i in range(num_files):
    start_idx = i * rows_per_file
    end_idx = min(start_idx + rows_per_file, total_rows)
    chunk = df.iloc[start_idx:end_idx]
    
    output_file = os.path.join(output_dir, f"part_{i+1:03d}.csv")
    chunk.to_csv(output_file, index=False)
    print(f"已保存: {output_file} （行数: {len(chunk)}）")

print("✅ 分割完成！")

正在读取文件: sample.csv
总行数: 100
已保存: split_csv_output_sample\part_001.csv （行数: 20）
已保存: split_csv_output_sample\part_002.csv （行数: 20）
已保存: split_csv_output_sample\part_003.csv （行数: 20）
已保存: split_csv_output_sample\part_004.csv （行数: 20）
已保存: split_csv_output_sample\part_005.csv （行数: 20）
✅ 分割完成！
